# Data Pipeline Incident Response — SFT → GRPO Training

**Meta PyTorch OpenEnv Hackathon · Round 2**

This notebook implements the full two-stage training pipeline:

1. **Stage 1 — SFT (Supervised Fine-Tuning):** warm-start the model on successful trajectories collected from Gemini 2.5 Flash on easy/medium tasks.
2. **Stage 2 — GRPO (Group Relative Policy Optimization):** use shaped environment rewards to improve performance on hard/hard2 schema-drift tasks.

Hardware: Kaggle T4 (16GB VRAM). Model: `unsloth/Meta-Llama-3.1-8B-Instruct`, 4-bit quantization.

Expected runtime: ~30 min SFT + ~60 min GRPO on a single T4.

## 1. Setup — Install dependencies and clone repo

In [ ]:
# Install Unsloth (fast fine-tuning) + TRL (GRPO) + environment dependencies
!pip install -q unsloth[colab-new] \
    trl>=0.8.6 \
    peft \
    accelerate \
    bitsandbytes \
    transformers \
    datasets \
    pandas numpy openai python-dotenv

print('Installation complete.')

In [ ]:
import os

# --- Kaggle: read GitHub token from secrets ---
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    GITHUB_TOKEN = _secrets.get_secret('GITHUB_TOKEN')
    HF_TOKEN     = _secrets.get_secret('HF_TOKEN')
except Exception:
    GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '')
    HF_TOKEN     = os.getenv('HF_TOKEN', '')

REPO_URL  = f'https://{GITHUB_TOKEN}@github.com/standing-on-giants/Meta_hackathon.git'
REPO_DIR  = '/kaggle/working/Meta_hackathon'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')

In [ ]:
import sys
import json
import textwrap
import random
import torch
import numpy as np
from pathlib import Path

sys.path.insert(0, REPO_DIR)
from src.environment import DataPipelineEnv
from src.models import PipelineAction, PipelineObservation

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Load model with Unsloth (4-bit quantization)

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 4096   # pipeline observations + action fit in ~2k tokens
DTYPE          = None   # auto-detect: bfloat16 on Ampere+, float16 on T4
LOAD_IN_4BIT   = True   # keeps model under 6GB on T4

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B-Instruct'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
    token           = HF_TOKEN,
)

print(f'Model loaded: {MODEL_NAME}')
print(f'Parameters:   {model.num_parameters() / 1e9:.2f}B')

In [ ]:
# Attach LoRA adapters — targets Q/K/V/O projections and FFN
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,      # LoRA rank
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha     = 16,
    lora_dropout   = 0.0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',  # saves ~30% VRAM
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.2f}%)')

## 3. Stage 1 — Supervised Fine-Tuning (SFT)

We collect successful trajectories from the environment using the Gemini-based oracle,
then fine-tune the model to imitate those trajectories. This gives the model a warm start
on action format, diagnostic strategy, and the pipeline observation structure.

In [ ]:
# Shared system prompt (same as inference.py)
SYSTEM_PROMPT = textwrap.dedent('''
You are an expert data engineer diagnosing and fixing broken data pipelines.

You will receive the current state of a pipeline (failing assertions, DAG structure,
historical run info) and must choose ONE action to take each turn.

WORKFLOW — always follow this order:
1. Read the failing assertions to understand what broke.
2. Use read_data_sample / check_schema / compare_schema to diagnose the ROOT CAUSE.
3. If compare_schema shows column renames → call handle_drift first.
4. Apply the fix using add_data_filter or patch_transformation.
5. Call run_pipeline to verify.
6. If data is unfixable, call alert_upstream_team.

CRITICAL: Never apply a fix before reading the data. Always respond with a single JSON object.
''').strip()


def format_obs(obs: PipelineObservation, step: int) -> str:
    """Convert observation to the text prompt shown to the model."""
    failed = '\n'.join(
        f'  [{r.assertion_id}] {r.assertion_type} on {r.table}({r.column or "N/A"}): {r.actual}'
        for r in obs.failed_assertions
    ) or '  (none)'
    passed = ', '.join(r.assertion_id for r in obs.passed_assertions) or 'none'
    dag    = '\n'.join(
        f'  {n.step_id}: {n.input_table} -> {n.output_table}'
        + (f' | filters: {n.applied_filters}' if n.applied_filters else '')
        + (f' | patches: {n.applied_patches}' if n.applied_patches else '')
        for n in obs.dag_structure
    )
    hist   = '\n'.join(f'  {r.date}: {r.status} ({r.row_count} rows)' for r in obs.historical_runs)
    schema = ''
    if obs.current_schema:
        schema += '\nCURRENT SCHEMA: ' + json.dumps(obs.current_schema)
    if obs.schema_diff:
        schema += '\nSCHEMA DIFF: ' + json.dumps(obs.schema_diff)
    sample = ''
    if obs.data_sample:
        sample = '\nDATA SAMPLE: ' + json.dumps(obs.data_sample[:3], default=str)

    return textwrap.dedent(f'''
    STEP {step}/{obs.max_steps} | TASK: {obs.task_id} ({obs.difficulty})
    DESCRIPTION: {obs.description}
    PIPELINE PASSED: {obs.pipeline_passed}
    LAST ACTION RESULT: {obs.last_action_result}

    DAG:\n{dag}
    FAILING:\n{failed}
    PASSING: {passed}
    HISTORY:\n{hist}
    RECENT ACTIONS:\n{chr(10).join("  " + a for a in obs.actions_taken[-4:]) or "  (none)"}
    {sample}{schema}
    Respond with exactly ONE action JSON object.
    ''').strip()


def make_chat_text(system: str, user: str, assistant: str) -> str:
    """Format a single (user, assistant) pair as a chat string for SFT."""
    return tokenizer.apply_chat_template(
        [
            {'role': 'system',    'content': system},
            {'role': 'user',      'content': user},
            {'role': 'assistant', 'content': assistant},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )

print('Prompt utilities defined.')

In [ ]:
# Gold-standard trajectories: each is (observation_text, action_json)
# These encode the correct diagnostic strategy for each task.

def collect_gold_trajectories(task_ids=None, n_episodes=5):
    """
    Run the environment with hard-coded gold actions per task to collect
    correct (observation, action) pairs for SFT.
    """
    if task_ids is None:
        task_ids = ['easy', 'medium']

    # Gold action sequences per task — these are the correct minimal fixes
    GOLD_ACTIONS = {
        'easy': [
            {'action_type': 'read_data_sample',      'params': {'table': 'raw_orders', 'n_rows': 20}},
            {'action_type': 'add_data_filter',       'params': {'step_id': 'transform_orders', 'filter_condition': 'user_id IS NOT NULL'}},
            {'action_type': 'run_pipeline',          'params': {}},
        ],
        'medium': [
            {'action_type': 'read_data_sample',      'params': {'table': 'raw_order_items', 'n_rows': 20}},
            {'action_type': 'patch_transformation',  'params': {'step_id': 'transform_items', 'patch_type': 'dedup', 'column': 'order_item_id'}},
            {'action_type': 'run_pipeline',          'params': {}},
        ],
    }

    pairs = []
    for task_id in task_ids:
        gold = GOLD_ACTIONS.get(task_id, [])
        if not gold:
            continue
        for _ in range(n_episodes):
            env = DataPipelineEnv(task_id=task_id)
            obs = env.reset()
            for step_idx, action_dict in enumerate(gold, 1):
                obs_text  = format_obs(obs, step_idx)
                action    = PipelineAction(**action_dict)
                pairs.append((obs_text, json.dumps(action_dict)))
                result    = env.step(action)
                obs       = result.observation
                if obs.pipeline_passed:
                    break
    return pairs


gold_pairs = collect_gold_trajectories(n_episodes=10)
print(f'Collected {len(gold_pairs)} (observation, action) pairs for SFT')
print('\nSample observation (truncated):')
print(gold_pairs[0][0][:400])
print('\nSample action:')
print(gold_pairs[0][1])

In [ ]:
# --- OPTIONAL: Collect trajectories using Gemini 2.5 Flash API ---
# This produces higher-quality SFT data than hard-coded gold actions
# because Gemini can handle edge cases and multi-step reasoning.
# Requires GEMINI_API_KEY in Kaggle secrets.

USE_GEMINI_TRAJECTORIES = False  # Set True to use Gemini API

if USE_GEMINI_TRAJECTORIES:
    from openai import OpenAI
    GEMINI_KEY = os.getenv('GEMINI_API_KEY', '')
    if not GEMINI_KEY:
        try: GEMINI_KEY = _secrets.get_secret('GEMINI_API_KEY')
        except: pass

    gemini_client = OpenAI(
        base_url='https://generativelanguage.googleapis.com/v1beta/openai/',
        api_key=GEMINI_KEY,
    )

    def gemini_collect(task_ids=['easy','medium'], n_ep=5):
        pairs = []
        for tid in task_ids:
            for ep in range(n_ep):
                env = DataPipelineEnv(task_id=tid)
                obs = env.reset()
                for step in range(1, 21):
                    if obs.pipeline_passed: break
                    prompt = format_obs(obs, step)
                    try:
                        resp = gemini_client.chat.completions.create(
                            model='gemini-2.5-flash',
                            messages=[
                                {'role':'system','content':SYSTEM_PROMPT},
                                {'role':'user','content':prompt},
                            ],
                            temperature=0.1, max_tokens=512,
                        )
                        action_text = resp.choices[0].message.content or ''
                    except Exception as e:
                        print(f'  Gemini error: {e}'); break
                    # Parse and execute
                    s = action_text.find('{')
                    e = action_text.rfind('}') + 1
                    if s == -1 or e <= s: break
                    try:
                        ad = json.loads(action_text[s:e])
                        action = PipelineAction(**ad)
                    except: break
                    pairs.append((prompt, json.dumps(ad)))
                    result = env.step(action)
                    obs = result.observation
                    if result.done: break
                nt = len(obs.failed_assertions) + len(obs.passed_assertions)
                np_ = len(obs.passed_assertions)
                score = np_/nt if nt > 0 else 0
                # Only keep high-quality trajectories
                if score < 0.8:
                    pairs = pairs[:-step]  # remove this episode
                    print(f'  {tid} ep{ep}: score={score:.2f} (dropped)')
                else:
                    print(f'  {tid} ep{ep}: score={score:.2f} (kept)')
        return pairs

    gemini_pairs = gemini_collect(n_ep=5)
    print(f'Gemini collected: {len(gemini_pairs)} pairs')
    # Merge with gold pairs
    gold_pairs = gold_pairs + gemini_pairs
    print(f'Total SFT pairs: {len(gold_pairs)}')
else:
    print('Skipping Gemini API collection (USE_GEMINI_TRAJECTORIES=False)')
    print(f'Using {len(gold_pairs)} gold trajectory pairs only')


In [ ]:
from datasets import Dataset

sft_texts = [
    make_chat_text(SYSTEM_PROMPT, obs_text, action_json)
    for obs_text, action_json in gold_pairs
]

sft_dataset = Dataset.from_dict({'text': sft_texts})
print(f'SFT dataset: {len(sft_dataset)} examples')
print(f'\nSample (first 600 chars):')
print(sft_dataset[0]['text'][:600])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

SFT_OUTPUT_DIR = '/kaggle/working/sft_checkpoint'

sft_trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    train_dataset  = sft_dataset,
    dataset_text_field = 'text',
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    args = TrainingArguments(
            report_to="none",
            average_tokens_across_devices=False,
        per_device_train_batch_size   = 2,
        gradient_accumulation_steps   = 4,    # effective batch = 8
        num_train_epochs              = 3,
        warmup_ratio                  = 0.1,
        learning_rate                 = 2e-4,
        fp16                          = not is_bfloat16_supported(),
        bf16                          = is_bfloat16_supported(),
        logging_steps                 = 5,
        optim                         = 'adamw_8bit',
        weight_decay                  = 0.01,
        lr_scheduler_type             = 'cosine',
        output_dir                    = SFT_OUTPUT_DIR,
        save_steps                    = 50,
        seed                          = 42,
    ),
)

print('Starting SFT training...')
sft_stats = sft_trainer.train()
print(f'SFT complete. Final loss: {sft_stats.training_loss:.4f}')
model.save_pretrained(SFT_OUTPUT_DIR)
tokenizer.save_pretrained(SFT_OUTPUT_DIR)
print(f'SFT checkpoint saved to {SFT_OUTPUT_DIR}')

## 4. Stage 2 — GRPO Fine-Tuning

GRPO uses the environment's shaped reward signal directly. For each prompt, we sample
G completions, score them against the environment, and update the policy to favour
higher-reward completions — no critic network needed.

KL penalty against the SFT reference prevents the policy from collapsing.

In [ ]:
# ------------------------------------------------------------------ #
# GRPO reward function
#
# Called once per completion. Takes the raw text output from the model
# and returns a float reward by executing that action in the environment.
#
# We maintain a per-prompt environment state so the same episode is
# reused across the G completions in a group.
# ------------------------------------------------------------------ #

import threading

_env_registry: dict = {}  # prompt_hash -> (env, obs, step_idx)
_env_lock = threading.Lock()


def get_or_create_env(prompt_hash: str, task_id: str):
    with _env_lock:
        if prompt_hash not in _env_registry:
            env = DataPipelineEnv(task_id=task_id)
            obs = env.reset()
            _env_registry[prompt_hash] = {'env': env, 'obs': obs, 'step': 1}
        return _env_registry[prompt_hash]


def pipeline_reward_fn(completions, prompts=None, task_ids=None, **kwargs) -> list[float]:
    """
    GRPO reward function.

    For each completion:
    - Parse the action JSON from the model output.
    - Execute it in the environment.
    - Return the immediate step reward.
    - Add a +0.3 bonus if the action JSON is syntactically valid.
    - Add a +0.5 bonus if the action is a non-blind fix (data was read first).
    """
    rewards = []
    for i, completion in enumerate(completions):
        text     = completion if isinstance(completion, str) else completion[0].get('content', '')
        task_id  = (task_ids or ['hard2'] * len(completions))[i]
        prompt_h = str(hash((task_id, i // 4)))  # group size G=4

        try:
            # Parse action from model output
            text = text.strip()
            if text.startswith('```'):
                text = '\n'.join(l for l in text.split('\n') if not l.startswith('```')).strip()
            start = text.find('{')
            end   = text.rfind('}') + 1
            if start == -1 or end == 0:
                rewards.append(-0.2)  # malformed output
                continue

            action_dict = json.loads(text[start:end])
            action      = PipelineAction(**action_dict)

            # Format reward: valid JSON is better than nothing
            format_bonus = 0.3

            # Execute in environment
            state  = get_or_create_env(prompt_h, task_id)
            result = state['env'].step(action)
            state['obs']  = result.observation
            state['step'] += 1

            env_reward = result.reward or 0.0

            # Schema-drift detection bonus: reward compare_schema after failed run_pipeline
            drift_bonus = 0.0
            if action.action_type == 'compare_schema':
                obs = result.observation
                if obs.schema_diff and len(obs.schema_diff) > 0:
                    drift_bonus = 0.3  # found real drift

            # Loop penalty: penalize repeated actions
            loop_penalty = 0.0
            obs_actions = result.observation.actions_taken or []
            if len(obs_actions) >= 3:
                last3 = [a.split('] ')[-1] if '] ' in a else a for a in obs_actions[-3:]]
                if last3[0] == last3[1] == last3[2]:
                    loop_penalty = -0.5

            total = format_bonus + env_reward + drift_bonus + loop_penalty
            rewards.append(float(total))

        except Exception as exc:
            rewards.append(-0.3)  # parse/execution error

    return rewards


print('Reward function defined.')

# Quick sanity check
_test_env = DataPipelineEnv(task_id='easy')
_test_obs = _test_env.reset()
_test_r   = pipeline_reward_fn(
    ['{"action_type": "read_data_sample", "params": {"table": "raw_orders", "n_rows": 10}}'],
    task_ids=['easy'],
)
print(f'Sanity check reward: {_test_r}  (should be > 0)')

In [ ]:
# GRPO training dataset: prompts only (no labels — GRPO generates its own completions)
# We use hard + hard2 tasks to push improvement on schema-drift scenarios.

def build_grpo_prompts(task_ids=None, n_per_task=30):
    if task_ids is None:
        task_ids = ['hard', 'hard2']
    prompts   = []
    task_meta = []

    for task_id in task_ids:
        for _ in range(n_per_task):
            env = DataPipelineEnv(task_id=task_id)
            obs = env.reset()
            prompt_text = format_obs(obs, step=1)
            chat_prompt = tokenizer.apply_chat_template(
                [
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': prompt_text},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            prompts.append({'prompt': chat_prompt})
            task_meta.append(task_id)

    return prompts, task_meta


grpo_prompts, grpo_task_ids = build_grpo_prompts(n_per_task=25)
grpo_dataset = Dataset.from_list(grpo_prompts)
print(f'GRPO prompt dataset: {len(grpo_dataset)} prompts')
print(f'Task distribution: hard={grpo_task_ids.count("hard")}, hard2={grpo_task_ids.count("hard2")}')

In [ ]:
from trl import GRPOConfig, GRPOTrainer

GRPO_OUTPUT_DIR = '/kaggle/working/grpo_checkpoint'

grpo_config = GRPOConfig(
    report_to="none",
    average_tokens_across_devices=False,
    output_dir                  = GRPO_OUTPUT_DIR,
    # Generation
    num_generations             = 4,    # G: sample 4 completions per prompt
    max_completion_length              = 200,  # actions are short JSON objects
    temperature                 = 0.8,  # higher temp for diverse sampling during training
    # Training
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_train_epochs            = 2,
    learning_rate               = 5e-5,
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    # GRPO-specific
    beta                    = 0.1,   # KL penalty against SFT reference
    loss_type                   = 'grpo',
    # Logging
    logging_steps               = 5,
    save_steps                  = 50,
    seed                        = 42,
    max_prompt_length              = MAX_SEQ_LENGTH,
    max_grad_norm               = 1.0,
    warmup_ratio                = 0.05,
)

grpo_trainer = GRPOTrainer(
    model         = model,
    tokenizer     = tokenizer,
    reward_funcs  = pipeline_reward_fn,
    args          = grpo_config,
    train_dataset = grpo_dataset,
)

print('Starting GRPO training...')
print(f'KL coefficient: {grpo_config.beta}')
print(f'Completions per prompt (G): {grpo_config.num_generations}')

grpo_stats = grpo_trainer.train()
print(f'GRPO complete. Final loss: {grpo_stats.training_loss:.4f}')

model.save_pretrained(GRPO_OUTPUT_DIR)
tokenizer.save_pretrained(GRPO_OUTPUT_DIR)
print(f'GRPO checkpoint saved to {GRPO_OUTPUT_DIR}')

## 5. Evaluation — Before vs After Comparison

In [ ]:
from unsloth import FastLanguageModel as FLM

def run_eval_episode(model, tokenizer, task_id: str, max_steps: int = 20) -> dict:
    """Run a single evaluation episode and return score metrics."""
    FLM.for_inference(model)  # enable fast inference mode

    env     = DataPipelineEnv(task_id=task_id)
    obs     = env.reset()
    rewards = []

    for step in range(1, max_steps + 1):
        if obs.pipeline_passed:
            break

        prompt_text = format_obs(obs, step)
        inputs = tokenizer.apply_chat_template(
            [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': prompt_text},
            ],
            return_tensors='pt',
            add_generation_prompt=True,
        ).to(model.device)

        with torch.no_grad():
            out_ids = model.generate(
                inputs,
                max_new_tokens=256,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        response = tokenizer.decode(out_ids[0][inputs.shape[1]:], skip_special_tokens=True).strip()

        # Parse action
        try:
            start = response.find('{')
            end   = response.rfind('}') + 1
            action = PipelineAction(**json.loads(response[start:end]))
        except Exception:
            action = PipelineAction(action_type='compare_schema', params={'table': 'raw_orders'})

        result  = env.step(action)
        obs     = result.observation
        rewards.append(result.reward or 0.0)

        if result.done:
            break

    n_total  = len(obs.failed_assertions) + len(obs.passed_assertions)
    n_passed = len(obs.passed_assertions)
    score    = min(max(n_passed / n_total if n_total > 0 else 0, 0.01), 0.99)

    return {
        'task_id':        task_id,
        'score':          round(score, 3),
        'pipeline_passed': obs.pipeline_passed,
        'total_reward':   round(sum(rewards), 3),
        'steps':          step,
    }

print('Evaluation function defined.')

In [ ]:
# Evaluate trained model on all tasks
EVAL_TASKS = ['easy', 'medium', 'hard', 'hard2']

print('Running post-training evaluation...')
trained_results = {}
for task_id in EVAL_TASKS:
    r = run_eval_episode(model, tokenizer, task_id=task_id)
    trained_results[task_id] = r
    status = '[PASSED]' if r['pipeline_passed'] else '[FAILED]'
    print(f'  {task_id:8s} | score={r["score"]:.3f} | reward={r["total_reward"]:+.2f} | steps={r["steps"]:2d} | {status}')

print(f'\nAvg score: {sum(r["score"] for r in trained_results.values()) / len(trained_results):.3f}')

In [ ]:
# Before/after comparison table
# Baseline scores from Gemini 2.5 Flash (pre-training reference)
BASELINE_SCORES = {
    'easy':   {'score': 0.99, 'pipeline_passed': True},
    'medium': {'score': 0.95, 'pipeline_passed': True},
    'hard':   {'score': 0.75, 'pipeline_passed': False},
    'hard2':  {'score': 0.30, 'pipeline_passed': False},  # untrained LLaMA baseline
}

print('\n' + '='*65)
print(f'{"Task":<10} {"Baseline":<12} {"Trained":<12} {"Delta":<10} {"Pass?"}')
print('='*65)
for task_id in EVAL_TASKS:
    base   = BASELINE_SCORES[task_id]['score']
    after  = trained_results[task_id]['score']
    delta  = after - base
    passed = '[PASSED]' if trained_results[task_id]['pipeline_passed'] else '[FAILED]'
    arrow  = '+' if delta >= 0 else ''
    print(f'{task_id:<10} {base:<12.3f} {after:<12.3f} {arrow}{delta:<9.3f} {passed}')
print('='*65)

In [ ]:
# Plot reward curves from GRPO training log
import matplotlib.pyplot as plt

if hasattr(grpo_trainer, 'state') and grpo_trainer.state.log_history:
    logs = [l for l in grpo_trainer.state.log_history if 'loss' in l]
    steps  = [l['step']  for l in logs]
    losses = [l['loss']  for l in logs]

    reward_logs = [l for l in grpo_trainer.state.log_history if 'rewards/mean' in l]
    r_steps  = [l['step']         for l in reward_logs]
    r_means  = [l['rewards/mean'] for l in reward_logs]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(steps, losses, 'b-', linewidth=1.5)
    ax1.set_title('GRPO Training Loss')
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Loss')
    ax1.grid(True, alpha=0.3)

    ax2.plot(r_steps, r_means, 'g-', linewidth=1.5)
    ax2.set_title('Mean Episode Reward (GRPO)')
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Reward')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/kaggle/working/reward_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Reward curve saved to /kaggle/working/reward_curve.png')
else:
    print('No training log available yet — run the GRPO training cell first.')

## 6. Save and push trained model to HuggingFace Hub

In [ ]:
HF_REPO = 'standing-on-giants/data-pipeline-incident-llama-grpo'

if HF_TOKEN:
    print(f'Pushing merged model to HF Hub: {HF_REPO}')
    model.push_to_hub_merged(
        HF_REPO,
        tokenizer,
        save_method = 'merged_16bit',
        token       = HF_TOKEN,
    )
    print(f'Model pushed to https://huggingface.co/{HF_REPO}')
else:
    print('HF_TOKEN not set — skipping Hub push. Model saved locally at:', GRPO_OUTPUT_DIR)

## 7. Schema Drift Demo — Before vs After

This cell runs a side-by-side demo showing the key story for the pitch:
**base model hallucinates the old column name; trained model calls compare_schema first.**

In [ ]:
print('=== SCHEMA DRIFT DEMO (hard2 task) ===')
print()
print('Scenario: After run 2, upstream API renames spend -> total_spend')
print('          Base model patches "spend" (column that no longer exists)')
print('          Trained model calls compare_schema first, detects the rename,')
print('          then calls handle_drift(resolve_column_rename)')
print()

# Simulate the critical decision point: step 4 after two run_pipeline calls
demo_env = DataPipelineEnv(task_id='hard2')
obs      = demo_env.reset()

# Fast-forward to post-drift state
for _ in range(3):
    r = demo_env.step(PipelineAction(action_type='run_pipeline', params={}))
    obs = r.observation

demo_prompt = format_obs(obs, step=4)

print('--- OBSERVATION AT STEP 4 (after drift applied) ---')
print(demo_prompt[:600])
print('...')
print()

# Generate trained model response
FLM.for_inference(model)
inputs = tokenizer.apply_chat_template(
    [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': demo_prompt},
    ],
    return_tensors='pt',
    add_generation_prompt=True,
).to(model.device)

with torch.no_grad():
    out_ids = model.generate(
        inputs,
        max_new_tokens=150,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

trained_response = tokenizer.decode(out_ids[0][inputs.shape[1]:], skip_special_tokens=True).strip()

print('--- BASE MODEL (hallucinated) ---')
print('{"action_type": "patch_transformation", "params": {"step_id": "transform_insights", "patch_type": "parse_currency", "column": "spend"}}')
print('  ^ patches column "spend" which no longer exists after rename drift')
print()
print('--- TRAINED MODEL ---')
print(trained_response)
print('  ^ should call compare_schema or handle_drift first')